In [3]:
import duckdb
import os
from pathlib import Path

# ensuring that os.chdir is idempotent and that we are in the project root directory,
# not inside the notebooks directory
if 'notebooks' not in os.listdir(Path.cwd()):
    print("Still inside notebooks directory, changing to project root directory.")
    os.chdir(Path.cwd().parent)
    print("Current working directory after change: ", Path.cwd())
else:
    print(f"Already in parent directory (current working directory: {Path.cwd()})")

Already in parent directory (current working directory: d:\space-weather-project-scrub)


In [11]:
# load config
from src.io.load_config import load_config
sw_config = load_config()['space_weather']

# load T1 and T2 dataset dir
T1_dir = sw_config['preprocessing']['k_index']['T1_output_dir']
T2_dir = sw_config['transform']['k_index']['T2_output_dir']

# Boilerplate code to fetch OMNI data

In [ ]:
import pandas as pd
import requests
from io import StringIO
# https://cdaweb.gsfc.nasa.gov/hapi/data?id=OMNI_HRO2_1MIN&parameters=F,BX_GSE,BY_GSM,BZ_GSM,flow_speed,proton_density,Pressure&time.min=2021-11-21T00:00:00Z&time.max=2021-11-22T00:00:00Z&format=csv
def fetch_omni_hro2_1min(start_utc: str, end_utc: str) -> pd.DataFrame:
    url = "https://cdaweb.gsfc.nasa.gov/hapi/data"
    params = {
        "id": "OMNI_HRO2_1MIN",
        "parameters": "F,BX_GSE,BY_GSM,BZ_GSM,flow_speed,proton_density,Pressure",
        "time.min": start_utc,
        "time.max": end_utc,
        #"format": "csv",
    }
    print("Requesting solar dataset from OMNI_HRO2_1MIN...")
    response = requests.get(url, params=params, timeout=120)
    response.raise_for_status()
    print("Request succeeded.")

    df = pd.read_csv(StringIO(response.text), comment="#", header=None)
    
    df.columns = [
        "time",
        "B_nT",
        "Bx_GSE_nT",
        "By_GSM_nT",
        "Bz_GSM_nT",
        "flow_speed_kms",
        "proton_density_ncc",
        "pressure_nPa",
    ]

    df["time"] = pd.to_datetime(df["time"], utc=True)

    fill_values = {
        "B_nT": 9999.99,
        "Bx_GSE_nT": 9999.99,
        "By_GSM_nT": 9999.99,
        "Bz_GSM_nT": 9999.99,
        "flow_speed_kms": 99999.9,
        "proton_density_ncc": 999.99,
        "pressure_nPa": 99.99,
    }

    for col, fill in fill_values.items():
        df.loc[df[col] == fill, col] = pd.NA

    return df

# Try fetching solar wind data to the HAPI

In [9]:
omni = fetch_omni_hro2_1min(
    "2021-11-21T00:00:00Z",
    "2021-11-22T00:00:00Z",
)

Requesting solar dataset from OMNI_HRO2_1MIN...
Request succeeded.


## Observations are recorded hourly (remember K-index observations are recorded every 3 hours)

In [14]:
omni.sort_values(by='time')

,time,B_nT,Bx_GSE_nT,By_GSM_nT,Bz_GSM_nT,flow_speed_kms,proton_density_ncc,pressure_nPa
0,2021-11-21 00:00:00+00:00,4.79,1.87,-2.40,-3.37,NaN,NaN,NaN
1,2021-11-21 00:01:00+00:00,5.73,4.25,0.29,2.60,NaN,NaN,NaN
2,2021-11-21 00:02:00+00:00,4.90,1.72,-3.04,-3.39,583.6,4.70,3.20
3,2021-11-21 00:03:00+00:00,4.92,2.03,-3.04,-3.23,583.6,4.70,3.20
4,2021-11-21 00:04:00+00:00,5.00,1.68,-2.93,-3.67,583.6,4.70,3.20
...,...,...,...,...,...,...,...,...
1435,2021-11-21 23:55:00+00:00,3.69,2.28,-2.76,0.48,612.4,2.33,1.75
1436,2021-11-21 23:56:00+00:00,3.76,1.76,-3.15,0.31,619.3,2.45,1.88
1437,2021-11-21 23:57:00+00:00,3.75,-0.65,-3.07,-1.14,635.0,2.35,1.90
1438,2021-11-21 23:58:00+00:00,3.95,-1.41,-3.01,-2.11,643.1,2.31,1.91
